<img src="../../images/logo/compactlogo.svg" width="200" alt="QENS">

# Tutorial 3: Decoder Comparison

QENS ships three decoders with different trade-offs between speed, memory, and decoding quality:

| Decoder | Algorithm | Time | Memory | Best For |
|---------|-----------|------|--------|----------|
| `LookupTableDecoder` | Precomputed table | O(1) | Exponential | d ≤ 7 |
| `MWPMDecoder` | Greedy min-weight matching | O(n²) | O(n) | d ≤ 15–20 |
| `UnionFindDecoder` | Growth + fusion | O(n·α(n)) | O(n) | Any size |

This notebook benchmarks all three on identical noise conditions so you can make an informed choice.

In [ ]:
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Remove for inline display

from qens import (
    RepetitionCode, SurfaceCode,
    DepolarizingError, BitFlipError,
    LookupTableDecoder, MWPMDecoder, UnionFindDecoder,
    NoisySampler, ThresholdExperiment,
)
from qens.core.types import PauliOp
from qens.viz.stats import plot_threshold, plot_logical_rates

## 1. Single-Point Comparison

First let's run all three decoders at a single (code, noise) operating point and compare their logical error rates.

In [ ]:
# Fixed operating point
code = RepetitionCode(distance=5)
noise = BitFlipError(p=0.05)
SHOTS = 10_000
SEED = 42

decoders = {
    "LookupTable": LookupTableDecoder(code),
    "MWPM": MWPMDecoder(code),
    "UnionFind": UnionFindDecoder(code),
}

sampler = NoisySampler(seed=SEED)

print(f"Code: {type(code).__name__}(d={code.code_distance}), Noise: {noise}, Shots: {SHOTS}")
print()
print(f"{'Decoder':>15s}  {'LER':>10s}  {'Logical Errs':>14s}  {'Precompute (s)':>16s}  {'Decode (s)':>12s}")
print("-" * 74)

results = {}
for name, decoder in decoders.items():
    # Time precompute
    t0 = time.perf_counter()
    decoder.precompute()
    t_pre = time.perf_counter() - t0

    # Time decode
    t0 = time.perf_counter()
    res = sampler.run(code, noise, decoder, shots=SHOTS)
    t_dec = time.perf_counter() - t0

    results[name] = res
    print(f"{name:>15s}  {res.logical_error_rate:>10.4f}  {res.logical_error_count:>14d}  {t_pre:>16.4f}  {t_dec:>12.4f}")

## 2. Precompute Time vs. Code Distance

Precompute time matters most for the lookup table decoder — its table grows exponentially with the number of stabilizers.

In [ ]:
distances = [3, 5, 7]
print(f"{'Distance':>10s}  {'LookupTable':>14s}  {'MWPM':>10s}  {'UnionFind':>12s}")
print("-" * 52)

for d in distances:
    c = RepetitionCode(distance=d)
    times = {}
    for cls, label in [(LookupTableDecoder, 'LookupTable'), (MWPMDecoder, 'MWPM'), (UnionFindDecoder, 'UnionFind')]:
        dec = cls(c)
        t0 = time.perf_counter()
        dec.precompute()
        times[label] = time.perf_counter() - t0
    print(f"{d:>10d}  {times['LookupTable']:>14.4f}s  {times['MWPM']:>8.4f}s  {times['UnionFind']:>10.4f}s")

## 3. Decoding Quality: Error Rate vs. Physical Noise

Compare how each decoder's logical error rate responds to increasing physical error rates.

In [ ]:
code_surf = SurfaceCode(distance=3)
error_rates = [0.005, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10]
SHOTS_SWEEP = 3_000

decoder_classes = [
    ("MWPM", MWPMDecoder),
    ("UnionFind", UnionFindDecoder),
]

print(f"Surface code d=3, {SHOTS_SWEEP} shots per point")
print()

header = f"{'p':>8s}" + "".join(f"  {n:>12s}" for n, _ in decoder_classes)
print(header)
print("-" * len(header))

sweep_results = {name: [] for name, _ in decoder_classes}

for p in error_rates:
    noise_p = DepolarizingError(p=p)
    sampler = NoisySampler(seed=0)
    row = f"{p:>8.4f}"
    for name, cls in decoder_classes:
        dec = cls(code_surf)
        res = sampler.run(code_surf, noise_p, dec, shots=SHOTS_SWEEP)
        sweep_results[name].append(res.logical_error_rate)
        row += f"  {res.logical_error_rate:>12.4f}"
    print(row)

## 4. Decoder Metadata: What Each Decoder Reveals

Each decoder's `DecoderResult.metadata` exposes decoder-specific information useful for debugging and visualization.

In [ ]:
surf3 = SurfaceCode(distance=3)
rng = np.random.default_rng(7)

noise_sample = DepolarizingError(p=0.05)
error = noise_sample.sample_errors(
    num_qubits=surf3.num_data_qubits,
    affected_qubits=list(range(surf3.num_data_qubits)),
    rng=rng,
)
syndrome = surf3.compute_syndrome(error)
print(f"Syndrome: {syndrome}")
print()

In [ ]:
# LookupTableDecoder metadata
# Only practical for small codes (d<=7)
rep3 = RepetitionCode(distance=3)
err_rep = np.zeros(3, dtype=np.uint8); err_rep[1] = PauliOp.X
syn_rep = rep3.compute_syndrome(err_rep)

lt_dec = LookupTableDecoder(rep3)
lt_res = lt_dec.decode(syn_rep)
print("=== LookupTableDecoder ===")
print(f"Syndrome:   {syn_rep}")
print(f"Correction: {lt_res.correction}")
print(f"Metadata:   {lt_res.metadata}")
print()

In [ ]:
# MWPMDecoder metadata
mwpm_dec = MWPMDecoder(surf3)
mwpm_dec.precompute()
mwpm_res = mwpm_dec.decode(syndrome)
print("=== MWPMDecoder ===")
print(f"Syndrome:    {syndrome}")
print(f"Correction:  {mwpm_res.correction}")
print(f"Num defects: {mwpm_res.metadata['num_defects']}")
print(f"Matching:    {mwpm_res.metadata['matching']}")
print()

In [ ]:
# UnionFindDecoder metadata
uf_dec = UnionFindDecoder(surf3)
uf_dec.precompute()
uf_res = uf_dec.decode(syndrome)
print("=== UnionFindDecoder ===")
print(f"Syndrome:    {syndrome}")
print(f"Correction:  {uf_res.correction}")
print(f"Num defects: {uf_res.metadata['num_defects']}")

## 5. Threshold Comparison

The most meaningful comparison is the **threshold plot** — how well each decoder suppresses logical errors as distance increases.

In [ ]:
# MWPM threshold sweep
exp_mwpm = ThresholdExperiment(
    code_class=SurfaceCode,
    distances=[3, 5],
    physical_error_rates=[0.005, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10],
    noise_model_factory=lambda p: DepolarizingError(p=p),
    decoder_class=MWPMDecoder,
    shots_per_point=1_000,
    seed=99,
)
result_mwpm = exp_mwpm.run()

fig_mwpm = plot_threshold(result_mwpm, title="Surface Code Threshold — MWPM Decoder")
fig_mwpm.show()
print("MWPM threshold plot rendered.")

In [ ]:
# UnionFind threshold sweep
exp_uf = ThresholdExperiment(
    code_class=SurfaceCode,
    distances=[3, 5],
    physical_error_rates=[0.005, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10],
    noise_model_factory=lambda p: DepolarizingError(p=p),
    decoder_class=UnionFindDecoder,
    shots_per_point=1_000,
    seed=99,
)
result_uf = exp_uf.run()

fig_uf = plot_threshold(result_uf, title="Surface Code Threshold — Union-Find Decoder")
fig_uf.show()
print("Union-Find threshold plot rendered.")

## 6. Decision Guide

Use this table to pick the right decoder for your use case:

| Use case | Recommended decoder | Reason |
|----------|--------------------|---------|
| Code distance ≤ 7, prototyping | `LookupTableDecoder` | Optimal corrections, O(1) lookup after precompute |
| Surface/color code, d ≤ 15 | `MWPMDecoder` | Near-optimal quality, reasonable speed |
| Large-scale threshold sweeps | `UnionFindDecoder` | Fastest per shot, sub-quadratic |
| Learning / debugging | `LookupTableDecoder` + small code | Full metadata, deterministic |
| Custom noise benchmarking | `MWPMDecoder` | Best correctible error range |

### Common pitfalls

- **Don't rely on `result.success`** from `decode()` for statistics — it checks the correction, not the residual. Use `NoisySampler.run()` instead.
- **`LookupTableDecoder` on d=9 surface code** will exhaust memory — the table has `2^(num_stabilizers)` entries.
- **`precompute()` is auto-called** on the first `decode()`, but calling it explicitly before a tight loop is good practice.

## Summary

```
LookupTableDecoder  →  optimal but exponential memory; use for small codes
MWPMDecoder         →  near-optimal; the go-to for surface code research
UnionFindDecoder    →  fastest per shot; ideal for large-scale sweeps
```

All three decoders follow the same interface:
```python
decoder = SomeDecoder(code)
decoder.precompute()              # optional but recommended
result = decoder.decode(syndrome) # returns DecoderResult
result.correction                 # PauliString to apply
result.metadata                   # decoder-specific info
```

See [Decoder Comparison Guide](../decoder-comparison.md) for a static reference version of this notebook.